# Week 2 — Data Collection and Cleaning

Cleaning data (monthly US exports and imports (from the US Census Bureau), exchange rates, GDP, inflation, and interest rates (all from the IMF)) and merging them all into a single master file.
**Final output:** `data/final/master_trade_flow.csv` — 198 countries, monthly data from 2017 to 2024.

## Step 1 — Clean Exports (`exports_monthly_raw.csv`)

In [1]:
import pandas as pd

exports_path = "/Users/angollapraveengoud/Trade_Flow_project/data/raw/exports_monthly_raw.csv"
df_exp = pd.read_csv(exports_path)

print("Shape:", df_exp.shape)
print("Columns:", df_exp.columns.tolist())

Shape: (36283, 6)
Columns: ['CTY_NAME', 'CTY_CODE', 'ALL_VAL_MO', 'time', 'year', 'month']


In [2]:
display(df_exp.head(5))
print("\nDtypes:")
print(df_exp.dtypes)
print("\nNulls in CTY_NAME:", df_exp["CTY_NAME"].isna().sum())

,CTY_NAME,CTY_CODE,ALL_VAL_MO,time,year,month
0,OPEC,0001,7650292366,2013-01,2013,1
1,EUROPEAN UNION,0003,20251001585,2013-01,2013,1
2,PACIFIC RIM COUNTRIES,0014,29292758959,2013-01,2013,1
3,CAFTA-DR,0017,2283973992,2013-01,2013,1
4,USMCA (NAFTA),0020,41073299616,2013-01,2013,1



Dtypes:
CTY_NAME        str
CTY_CODE        str
ALL_VAL_MO    int64
time            str
year          int64
month         int64
dtype: object

Nulls in CTY_NAME: 13


In [3]:
before_rows = len(df_exp)

mask_bad_code = (
    df_exp["CTY_CODE"].astype(str).str.contains("X", na=False) |
    (df_exp["CTY_CODE"].astype(str) == "-") |
    df_exp["CTY_CODE"].astype(str).str.startswith("0", na=False)
)

df_exp = df_exp[~mask_bad_code].copy()
print("Rows before:", before_rows, "| after code-filter:", len(df_exp))

Rows before: 36283 | after code-filter: 33331


In [4]:
before_null_filter = len(df_exp)
df_exp = df_exp.dropna(subset=["CTY_NAME"]).copy()

df_exp["ALL_VAL_MO"] = pd.to_numeric(df_exp["ALL_VAL_MO"], errors="coerce")
df_exp = df_exp.dropna(subset=["ALL_VAL_MO"]).copy()

print("After CTY_NAME null filter:", len(df_exp), "(dropped", before_null_filter - len(df_exp), ")")
print("ALL_VAL_MO dtype:", df_exp["ALL_VAL_MO"].dtype)

After CTY_NAME null filter: 33318 (dropped 13 )
ALL_VAL_MO dtype: int64


In [5]:
df_exp = df_exp.rename(columns={
    "CTY_NAME": "country",
    "CTY_CODE": "country_code",
    "ALL_VAL_MO": "exports_usd"
})

df_exp = df_exp[["country", "country_code", "exports_usd", "year", "month"]].copy()
df_exp["country"] = df_exp["country"].astype(str).str.title()

In [6]:
# Save cleaned exports
df_exp.to_csv("../data/cleaned/exports_cleaned.csv", index=False)
print("Final shape:", df_exp.shape)
print("Unique countries:", df_exp["country"].nunique())
print("Unique year-month combos:", df_exp[["year", "month"]].drop_duplicates().shape[0])

Final shape: (33318, 5)
Unique countries: 234
Unique year-month combos: 144


## Step 2 — Clean Imports (`imports_monthly_raw.csv`)

In [7]:
imports_path = "../data/raw/imports_monthly_raw.csv"
df_imp = pd.read_csv(imports_path)

print("Shape:", df_imp.shape)
print("Columns:", df_imp.columns.tolist())
display(df_imp.head(5))
print(df_imp.dtypes)

Shape: (36277, 6)
Columns: ['CTY_NAME', 'CTY_CODE', 'GEN_VAL_MO', 'time', 'year', 'month']


,CTY_NAME,CTY_CODE,GEN_VAL_MO,time,year,month
0,TOTAL FOR ALL COUNTRIES,-,185168566967,2013-01,2013,1
1,PAKISTAN,5350,330146080,2013-01,2013,1
2,NEPAL,5360,7006110,2013-01,2013,1
3,BANGLADESH,5380,493761700,2013-01,2013,1
4,SRI LANKA,5420,212440298,2013-01,2013,1


CTY_NAME        str
CTY_CODE        str
GEN_VAL_MO    int64
time            str
year          int64
month         int64
dtype: object


In [8]:
before_rows = len(df_imp)

mask_bad_code = (
    df_imp["CTY_CODE"].astype(str).str.contains("X", na=False) |
    (df_imp["CTY_CODE"].astype(str) == "-") |
    df_imp["CTY_CODE"].astype(str).str.startswith("0", na=False)
)

df_imp = df_imp[~mask_bad_code].copy()
print("Rows before:", before_rows, "| after code-filter:", len(df_imp))

Rows before: 36277 | after code-filter: 33325


In [9]:
before_null_filter = len(df_imp)
df_imp = df_imp.dropna(subset=["CTY_NAME"]).copy()

df_imp["GEN_VAL_MO"] = pd.to_numeric(df_imp["GEN_VAL_MO"], errors="coerce")
df_imp = df_imp.dropna(subset=["GEN_VAL_MO"]).copy()

print("After CTY_NAME null filter:", len(df_imp), "(dropped", before_null_filter - len(df_imp), ")")
print("GEN_VAL_MO dtype:", df_imp["GEN_VAL_MO"].dtype)

After CTY_NAME null filter: 33325 (dropped 0 )
GEN_VAL_MO dtype: int64


In [10]:
df_imp = df_imp.rename(columns={
    "CTY_NAME": "country",
    "CTY_CODE": "country_code",
    "GEN_VAL_MO": "imports_usd"
})

df_imp = df_imp[["country", "country_code", "imports_usd", "year", "month"]].copy()
df_imp["country"] = df_imp["country"].astype(str).str.title()
df_imp.to_csv("../data/cleaned/imports_cleaned.csv", index=False)

print("Final shape:", df_imp.shape)
print("Unique countries:", df_imp["country"].nunique())
print("Unique year-month combos:", df_imp[["year", "month"]].drop_duplicates().shape[0])


Final shape: (33325, 5)
Unique countries: 234
Unique year-month combos: 144


## Step 3 — Clean Exchange Rates (`exchange_rate_monthly_raw.csv`)

In [11]:
er_path = "../data/raw/exchange_rate_monthly_raw.csv"
df_er = pd.read_csv(er_path, encoding="latin1", low_memory=False)

print("Shape:", df_er.shape)
print("Columns count:", len(df_er.columns))
print(df_er[["FREQUENCY", "INDICATOR", "TYPE_OF_TRANSFORMATION"]].head(5))

Shape: (10738, 1773)
Columns count: 1773
   FREQUENCY                  INDICATOR TYPE_OF_TRANSFORMATION
0  Quarterly  ECU per domestic currency         Period average
1     Annual  ECU per domestic currency    End-of-period (EoP)
2    Monthly  Domestic currency per ECU         Period average
3  Quarterly  ECU per domestic currency         Period average
4     Annual  ECU per domestic currency         Period average


In [12]:
df_er_f = df_er[df_er["FREQUENCY"] == "Monthly"].copy()
print("After Monthly filter:", df_er_f.shape)

df_er_f = df_er_f[df_er_f["INDICATOR"].str.contains("Domestic currency per US Dollar", na=False)].copy()
print("After USD indicator filter:", df_er_f.shape)

df_er_f = df_er_f[df_er_f["TYPE_OF_TRANSFORMATION"] == "Period average"].copy()
print("After Period average filter:", df_er_f.shape)

After Monthly filter: (3582, 1773)
After USD indicator filter: (454, 1773)
After Period average filter: (227, 1773)


In [13]:
month_cols = [c for c in df_er_f.columns if isinstance(c, str) and "-" in c and c.count("-") == 1 and c.split("-")[1].startswith("M")]
print("Month columns found:", len(month_cols), "| sample:", month_cols[:5])

df_er_long = df_er_f.melt(
    id_vars=["COUNTRY"],
    value_vars=month_cols,
    var_name="time_period",
    value_name="exchange_rate"
)
print("Long shape:", df_er_long.shape)

Month columns found: 1226 | sample: ['1924-M01', '1924-M02', '1924-M03', '1924-M04', '1924-M05']
Long shape: (278302, 3)


In [14]:
df_er_long["year"] = pd.to_numeric(df_er_long["time_period"].str.split("-").str[0], errors="coerce")
df_er_long["month"] = pd.to_numeric(df_er_long["time_period"].str.split("-").str[1].str.replace("M", "", regex=False), errors="coerce")
df_er_long["exchange_rate"] = pd.to_numeric(df_er_long["exchange_rate"], errors="coerce")

df_er_long = df_er_long[df_er_long["year"].between(2013, 2024)].copy()
df_er_long = df_er_long.dropna(subset=["exchange_rate", "COUNTRY", "year", "month"]).copy()
print("After year+NaN filter:", df_er_long.shape)

After year+NaN filter: (27229, 5)


In [15]:
df_er_clean = df_er_long.rename(columns={"COUNTRY": "country"})[["country", "year", "month", "exchange_rate"]].copy()
df_er_clean["country"] = df_er_clean["country"].astype(str).str.title()
df_er_clean = df_er_clean.drop_duplicates(subset=["country", "year", "month"]).copy()

print("Final ER shape:", df_er_clean.shape)
print("Unique countries:", df_er_clean["country"].nunique())
print("Unique year-month combos:", df_er_clean[["year", "month"]].drop_duplicates().shape[0])

df_er_clean.to_csv("../data/cleaned/exchange_rate_cleaned.csv", index=False)


Final ER shape: (27229, 4)
Unique countries: 195
Unique year-month combos: 144


## Step 4 — Clean GDP (`imf_res_weo_gdp_wide.csv`)

In [16]:
gdp_path = "../data/raw/imf_res_weo_gdp_wide.csv"
df_gdp = pd.read_csv(gdp_path, encoding="latin1", low_memory=False)

print("Shape:", df_gdp.shape)
print("Columns:", df_gdp.columns.tolist())
display(df_gdp[["COUNTRY", "2017", "2018", "2024", "2025", "2026"]].head(5))

Shape: (209, 17)
Columns: ['ï»¿"DATASET"', 'SERIES_CODE', 'OBS_MEASURE', 'COUNTRY', 'INDICATOR', 'FREQUENCY', 'SCALE', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']


,COUNTRY,2017,2018,2024,2025,2026
0,Advanced Economies,49016.624,51868.982,64901.125,68598.619,72202.717
1,United Kingdom,2682.384,2875.024,3644.636,3958.780,4225.639
2,G7,37196.592,39238.227,49422.823,52056.759,54519.693
3,United States,19612.100,20656.525,29298.025,30615.743,31821.293
4,Germany,3764.023,4057.272,4684.182,5013.574,5328.184


In [17]:
before_rows = len(df_gdp)

agg_pattern = r"Advanced Economies|^G7$|Euro Area|^World$|ASEAN-5|European Union|Emerging Market and Developing Economies|Latin America and the Caribbean|Middle East and Central Asia|Emerging and Developing Asia|Sub-Saharan Africa|Emerging and Developing Europe|Other Advanced"
df_gdp = df_gdp[~df_gdp["COUNTRY"].str.contains(agg_pattern, case=False, na=False)].copy()

print("Rows before:", before_rows, "| after aggregate filter:", len(df_gdp))
print("Sample countries:", df_gdp["COUNTRY"].head(10).to_list())

Rows before: 209 | after aggregate filter: 196
Sample countries: ['United Kingdom', 'United States', 'Germany', 'Austria', 'Belgium', 'Italy', 'Denmark', 'France', 'Netherlands, The', 'Luxembourg']


In [18]:
year_cols = [str(y) for y in range(2017, 2025)]
df_gdp = df_gdp[["COUNTRY"] + year_cols].copy()

df_gdp_long = df_gdp.melt(
    id_vars=["COUNTRY"],
    value_vars=year_cols,
    var_name="year",
    value_name="gdp_billions"
)

print("GDP long shape:", df_gdp_long.shape)
display(df_gdp_long.head(5))

GDP long shape: (1568, 3)


,COUNTRY,year,gdp_billions
0,United Kingdom,2017,2682.384
1,United States,2017,19612.100
2,Germany,2017,3764.023
3,Austria,2017,414.780
4,Belgium,2017,500.733


In [19]:
df_gdp_long["year"] = pd.to_numeric(df_gdp_long["year"], errors="coerce").astype("Int64")
df_gdp_long["gdp_billions"] = pd.to_numeric(df_gdp_long["gdp_billions"], errors="coerce")

df_gdp_clean = df_gdp_long.dropna(subset=["year", "gdp_billions"]).rename(columns={"COUNTRY": "country"})
df_gdp_clean["country"] = df_gdp_clean["country"].astype(str).str.title()
# WEO uses "Netherlands, The"; Census/ER use "Netherlands" — align for merge
df_gdp_clean["country"] = df_gdp_clean["country"].replace({"Netherlands, The": "Netherlands"})
df_gdp_clean = df_gdp_clean[["country", "year", "gdp_billions"]].copy()

df_gdp_clean.to_csv("../data/cleaned/gdp_cleaned.csv", index=False)


## Step 6 — Clean Interest Rates (`interest_rate_monthly_raw.csv`)


In [20]:
ir_path = "../data/raw/interest_rate_monthly_raw.csv"
hdr = pd.read_csv(ir_path, nrows=0, encoding="latin1").columns.tolist()

meta = ["COUNTRY", "INDICATOR", "FREQUENCY"]
mcols = [c for c in hdr if isinstance(c, str) and "-" in c and c.count("-") == 1 and c.split("-")[1].startswith("M")]

df_ir = pd.read_csv(ir_path, usecols=meta + mcols, encoding="latin1", engine="python", on_bad_lines="skip")
print("Loaded shape:", df_ir.shape, "| month columns:", len(mcols))
print("\nFREQUENCY:\n", df_ir["FREQUENCY"].value_counts())
print("\nINDICATOR (top 12):\n", df_ir["INDICATOR"].value_counts().head(12))

Loaded shape: (3395, 918) | month columns: 915

FREQUENCY:
 FREQUENCY
Monthly      1140
Quarterly    1128
Annual       1127
Name: count, dtype: int64

INDICATOR (top 12):
 INDICATOR
Deposit Rate, Percent per annum                                                                                                       426
Lending Rate, Percent per annum                                                                                                       408
Money market Rate, Percent per annum                                                                                                  276
Government securities: Treasury bills yields, Rate, Percent per annum                                                                 261
Monetary policy-related, Rate, Percent per annum                                                                                      249
Discount Rate, Percent per annum                                                                                                

In [21]:
df_ir2 = df_ir[
    (df_ir["FREQUENCY"] == "Monthly")
    & (df_ir["INDICATOR"] == "Lending Rate, Percent per annum")
].copy()
print("After Monthly + Lending Rate filter:", df_ir2.shape)

After Monthly + Lending Rate filter: (136, 918)


In [22]:
vcols = [c for c in df_ir2.columns if c not in ["COUNTRY", "INDICATOR", "FREQUENCY"]]
df_irl = df_ir2.melt(
    id_vars=["COUNTRY"],
    value_vars=vcols,
    var_name="time_period",
    value_name="interest_rate"
)
print("Long shape:", df_irl.shape)

Long shape: (124440, 3)


In [23]:
df_irl["year"] = pd.to_numeric(df_irl["time_period"].str.split("-").str[0], errors="coerce")
df_irl["month"] = pd.to_numeric(df_irl["time_period"].str.split("-").str[1].str.replace("M", "", regex=False), errors="coerce")
df_irl["interest_rate"] = pd.to_numeric(df_irl["interest_rate"], errors="coerce")

df_irc = df_irl[df_irl["year"].between(2013, 2024)].dropna(subset=["interest_rate", "COUNTRY", "year", "month"]).copy()
df_irc = df_irc.rename(columns={"COUNTRY": "country"})[["country", "year", "month", "interest_rate"]].copy()
df_irc["country"] = df_irc["country"].astype(str).str.title()
df_irc["country"] = df_irc["country"].replace({"Netherlands, The": "Netherlands"})
df_irc = df_irc.drop_duplicates(subset=["country", "year", "month"])

df_irc.to_csv("../data/cleaned/interest_rate_cleaned.csv", index=False)


## Step 7 — Country Name Mapping

Census trade data and IMF macro files use different country names (e.g. Census says "China", IMF says "China, People'S Republic Of"). We build a mapping dictionary, verify each target against the exchange-rate file, and apply it to exports/imports before merging.

**Result:** 74 verified mappings saved to `data/country_name_mapping.csv`.

In [24]:
from pathlib import Path
import pandas as pd

def project_root() -> Path:
    marker = Path("data/cleaned/exports_cleaned.csv")
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / marker).is_file():
            return d
    return Path("/Users/angollapraveengoud/Trade_Flow_project")

base = project_root()
print("PROJECT_ROOT =", base.resolve())

PROJECT_ROOT = /Users/angollapraveengoud/Desktop/ONE_LAST!


In [25]:
exp = pd.read_csv(base / "data/cleaned/exports_cleaned.csv")
imp = pd.read_csv(base / "data/cleaned/imports_cleaned.csv")
er = pd.read_csv(base / "data/cleaned/exchange_rate_cleaned.csv")
er_set = set(er["country"])
trade_set = set(exp["country"]) | set(imp["country"])
print("trade:", len(trade_set), "| ER:", er["country"].nunique())

trade: 234 | ER: 195


### 7a — Full `trade_to_imf` dictionary (all verified rounds)

Every value below was checked with `value in er_set → True` before inclusion.

In [26]:
trade_to_imf = {
    # Batch 1 — major partners
    "Burma": "Myanmar",
    "China": "China, People'S Republic Of",
    "Korea, South": "Korea, Republic Of",
    "Bahamas": "Bahamas, The",
    "Brunei": "Brunei Darussalam",
    "Hong Kong": "Hong Kong Special Administrative Region, People'S Republic Of China",
    "Macau": "Macao Special Administrative Region, People'S Republic Of China",
    "Gambia": "Gambia, The",
    "Eswatini": "Eswatini, Kingdom Of",
    "Democratic Republic Of The Congo": "Congo, Democratic Republic Of The",
    "Cote D'Ivoire": "Cã´Te D'Ivoire",
    "Curacao": "Curaã§Ao, Kingdom Of The Netherlands",
    "Congo": "Congo, Republic Of",
    # Batch 2 — formal IMF names
    "Afghanistan": "Afghanistan, Islamic Republic Of",
    "Andorra": "Andorra, Principality Of",
    "Anguilla": "Anguilla, United Kingdom-British Overseas Territory",
    "Armenia": "Armenia, Republic Of",
    "Aruba": "Aruba, Kingdom Of The Netherlands",
    "Azerbaijan": "Azerbaijan, Republic Of",
    "Bahrain": "Bahrain, Kingdom Of",
    "Belarus": "Belarus, Republic Of",
    "Comoros": "Comoros, Union Of The",
    "Egypt": "Egypt, Arab Republic Of",
    "Equatorial Guinea": "Equatorial Guinea, Republic Of",
    "Eritrea": "Eritrea, The State Of",
    "Ethiopia": "Ethiopia, The Federal Democratic Republic Of",
    # Batch 3 — verified new_mappings
    "Fiji": "Fiji, Republic Of",
    "Iran": "Iran, Islamic Republic Of",
    "Kazakhstan": "Kazakhstan, Republic Of",
    "Moldova": "Moldova, Republic Of",
    "Serbia": "Serbia, Republic Of",
    "South Sudan": "South Sudan, Republic Of",
    "Syria": "Syrian Arab Republic",
    "Tanzania": "Tanzania, United Republic Of",
    "Yemen": "Yemen, Republic Of",
    "Micronesia": "Micronesia, Federated States Of",
    "Kosovo": "Kosovo, Republic Of",
    "Lesotho": "Lesotho, Kingdom Of",
    "Madagascar": "Madagascar, Republic Of",
    "Mauritania": "Mauritania, Islamic Republic Of",
    "Mozambique": "Mozambique, Republic Of",
    "Tajikistan": "Tajikistan, Republic Of",
    "Uzbekistan": "Uzbekistan, Republic Of",
    # Batch 4 — fix_round2 (short IMF names)
    "Laos": "Lao People'S Democratic Republic",
    "Libya": "Libya",
    "Somalia": "Somalia",
    "Timor-Leste": "Timor-Leste, Democratic Republic Of",
    "Venezuela": "Venezuela, Repãºblica Bolivariana De",
    "Vietnam": "Vietnam",
    "Zambia": "Zambia",
    "Zimbabwe": "Zimbabwe",
    "Bolivia": "Bolivia",
    "Cabo Verde": "Cabo Verde",
    "Malawi": "Malawi",
    "Seychelles": "Seychelles",
    "Sierra Leone": "Sierra Leone",
    "Suriname": "Suriname",
    "Tonga": "Tonga",
    # Batch 5 — fix_round3 (encoding + formal)
    "Liechtenstein": "Liechtenstein, Principality Of",
    "Macedonia": "North Macedonia, Republic Of",
    "Montserrat": "Montserrat, United Kingdom-British Overseas Territory",
    "Nauru": "Nauru, Republic Of",
    "Palau": "Palau, Republic Of",
    "Poland": "Poland, Republic Of",
    "Russia": "Russian Federation",
    "San Marino": "San Marino, Republic Of",
    "Sint Maarten": "Sint Maarten, Kingdom Of The Netherlands",
    "Taiwan": "Taiwan Province Of China",
    "Kyrgyzstan": "Kyrgyz Republic",
    "Turkey": "Tã¼Rkiye, Republic Of",
    "St Kitts And Nevis": "St. Kitts And Nevis",
    "St Lucia": "St. Lucia",
    "St Vincent And The Grenadines": "St. Vincent And The Grenadines",
    "Sao Tome And Principe": "Sã£O Tomã© And Prã\xadNcipe, Democratic Republic Of",
}

print("Total mappings:", len(trade_to_imf))

Total mappings: 74


In [27]:
exp["country_imf"] = exp["country"].map(trade_to_imf).fillna(exp["country"])
imp["country_imf"] = imp["country"].map(trade_to_imf).fillna(imp["country"])
trade_imf = set(exp["country_imf"]) | set(imp["country_imf"])
still_missing = sorted(trade_imf - er_set)
print("Still missing from ER:", len(still_missing))

Still missing from ER: 51


In [28]:
pd.DataFrame(
    list(trade_to_imf.items()),
    columns=["trade_name", "imf_name"],
).to_csv(base / "data/country_name_mapping.csv", index=False)


## Step 8 — Euro-area Exchange Rates

Euro-zone countries (France, Germany, etc.) have no national-currency-per-USD in the IMF extract for 2013–2024. We copy the **Euro Area (Ea)** EUR/USD series to all 20 euro members so they get exchange-rate data after the merge.

**Note:** Croatia adopted the Euro in Jan 2023; using EUR/USD for all months is a simplifying assumption.

In [29]:
er_all = pd.read_csv(base / "data/cleaned/exchange_rate_cleaned.csv")
euro_template = er_all.loc[er_all["country"] == "Euro Area (Ea)", ["year", "month", "exchange_rate"]]
print("EUR/USD template months:", len(euro_template))

euro_members = [
    "France", "Germany", "Italy", "Spain", "Netherlands", "Belgium", "Austria",
    "Ireland", "Finland", "Portugal", "Greece", "Luxembourg", "Slovakia",
    "Slovenia", "Estonia", "Latvia", "Lithuania", "Malta", "Cyprus", "Croatia",
]

chunks = []
for name in euro_members:
    d = euro_template.copy()
    d["country"] = name
    chunks.append(d)

er_euro = pd.concat(chunks, ignore_index=True)
er_new = pd.concat([er_all, er_euro], ignore_index=True)
er_new.to_csv(base / "data/cleaned/exchange_rate_cleaned.csv", index=False)
print("ER rows:", len(er_new), "| France in ER:", "France" in set(er_new["country"]))

EUR/USD template months: 144
ER rows: 30109 | France in ER: True


## Step 9 — Merge All Datasets into Master Panel

1. Reload all cleaned files (ER now includes euro rows).
2. Expand annual GDP to 12 monthly rows per country-year.
3. Outer-join exports + imports on `country_imf, year, month`.
4. Left-join ER, GDP, inflation, interest rates.
5. Filter to **2017–2024** and save `data/final/master_trade_flow.csv`.

In [30]:
er = pd.read_csv(base / "data/cleaned/exchange_rate_cleaned.csv")
inf = pd.read_csv(base / "data/cleaned/inflation_cleaned.csv")
ir = pd.read_csv(base / "data/cleaned/interest_rate_cleaned.csv")
gdp = pd.read_csv(base / "data/cleaned/gdp_cleaned.csv")
gdp_m = gdp.merge(pd.DataFrame({"month": range(1, 13)}), how="cross")
print("Shapes — ER:", er.shape, "| GDP monthly:", gdp_m.shape, "| Inflation:", inf.shape, "| Interest:", ir.shape)

Shapes — ER: (30109, 4) | GDP monthly: (18756, 4) | Inflation: (18756, 4) | Interest: (16115, 4)


In [31]:
m = exp.merge(imp, on=["country_imf", "year", "month"], how="outer", suffixes=("_exp", "_imp"))
m = m.merge(er, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])
m = m.merge(gdp_m, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])
m = m.merge(inf, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])
m = m.merge(ir, left_on=["country_imf", "year", "month"], right_on=["country", "year", "month"], how="left").drop(columns=["country"])

master = m[(m["year"] >= 2017) & (m["year"] <= 2024)].copy()
(base / "data/final").mkdir(parents=True, exist_ok=True)
master.to_csv(base / "data/final/master_trade_flow.csv", index=False)
print("master:", master.shape)
print("ER null %:", round(100 * master["exchange_rate"].isna().mean(), 1))
print("duplicates:", master.duplicated(subset=["country_imf", "year", "month"]).sum())

master: (22306, 13)
ER null %: 14.8
duplicates: 0


## Step 10 — Data Quality Report

Summary statistics, null percentages, top trade partners, and spot checks.

In [32]:
print("MASTER DATASET")
print("Shape:", master.shape)
print("Year range:", master["year"].min(), "–", master["year"].max())
print("Unique partners:", master["country_imf"].nunique())
print("Unique year-months:", master[["year", "month"]].drop_duplicates().shape[0])
print("\nNULL % PER COLUMN")
print((master.isna().mean() * 100).round(1).sort_values(ascending=False))
print("\nTOP 5 EXPORT PARTNERS")
print(master.groupby("country_imf")["exports_usd"].sum().sort_values(ascending=False).head(5))
print("\nTOP 5 IMPORT PARTNERS")
print(master.groupby("country_imf")["imports_usd"].sum().sort_values(ascending=False).head(5))

MASTER DATASET
Shape: (22306, 13)
Year range: 2017 – 2024
Unique partners: 234
Unique year-months: 96

NULL % PER COLUMN
interest_rate       56.1
gdp_billions        20.2
inflation           18.5
exchange_rate       14.8
country_exp          0.6
country_code_exp     0.6
exports_usd          0.6
country_imp          0.2
country_code_imp     0.2
imports_usd          0.2
year                 0.0
month                0.0
country_imf          0.0
dtype: float64

TOP 5 EXPORT PARTNERS
country_imf
Canada                         2.504542e+12
Mexico                         2.240646e+12
China, People'S Republic Of    1.077630e+12
Japan                          5.909311e+11
United Kingdom                 5.434071e+11
Name: exports_usd, dtype: float64

TOP 5 IMPORT PARTNERS
country_imf
China, People'S Republic Of    3.831841e+12
Mexico                         3.148933e+12
Canada                         2.830844e+12
Japan                          1.120116e+12
Germany                        1.086432

In [33]:
for c in ["China, People'S Republic Of", "France", "Fiji, Republic Of", "Myanmar"]:
    sub = master[master["country_imf"] == c]
    er_pct = sub["exchange_rate"].notna().mean() if len(sub) > 0 else 0
    print(f"{c}: rows={len(sub)} | ER={er_pct:.0%}")

China, People'S Republic Of: rows=96 | ER=100%
France: rows=96 | ER=100%
Fiji, Republic Of: rows=96 | ER=100%
Myanmar: rows=96 | ER=54%
